In [2]:
# 4_model_training.ipynb
"""
Customer Churn Analysis - Model Training
Task 4 & 5: Choose suitable binary classification algorithm and train model
"""

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import cross_val_score
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("TASKS 4 & 5: MODEL SELECTION & TRAINING")
print("Binary Classification Algorithms for Churn Prediction")
print("="*80)

# Step 1: Load data
print("\n📂 STEP 1: LOADING TRAINING DATA")
print("-"*50)

X_train = pd.read_csv('data/processed/X_train.csv')
X_test = pd.read_csv('data/processed/X_test.csv')
y_train = pd.read_csv('data/processed/y_train.csv')['Churn']
y_test = pd.read_csv('data/processed/y_test.csv')['Churn']

print(f"✅ Data loaded successfully!")
print(f"   Training set: {X_train.shape}")
print(f"   Testing set: {X_test.shape}")

# Step 2: Handle class imbalance with SMOTE
print("\n⚖️ STEP 2: HANDLING CLASS IMBALANCE")
print("-"*50)

print(f"Before SMOTE:")
print(f"   Churn: {(y_train==1).sum()} ({(y_train==1).mean()*100:.1f}%)")
print(f"   No Churn: {(y_train==0).sum()} ({(y_train==0).mean()*100:.1f}%)")

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE:")
print(f"   Churn: {(y_train_resampled==1).sum()} ({(y_train_resampled==1).mean()*100:.1f}%)")
print(f"   No Churn: {(y_train_resampled==0).sum()} ({(y_train_resampled==0).mean()*100:.1f}%)")

# Step 3: Define models
print("\n🤖 STEP 3: SELECTING BINARY CLASSIFICATION ALGORITHMS")
print("-"*50)

models = {
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=1000, random_state=42),
        'description': 'Baseline model, interpretable, good for linear relationships'
    },
    'Random Forest': {
        'model': RandomForestClassifier(n_estimators=200, max_depth=15, 
                                        min_samples_split=5, random_state=42),
        'description': 'Ensemble method, handles non-linear relationships, feature importance'
    },
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                            max_depth=5, random_state=42),
        'description': 'Boosted trees, often best performance, handles complex patterns'
    },
    'SVM': {
        'model': SVC(kernel='rbf', probability=True, random_state=42),
        'description': 'Powerful for high-dimensional spaces, effective with kernel tricks'
    }
}

# Step 4: Train and evaluate models
print("\n🚀 STEP 4: TRAINING MODELS")
print("="*80)

results = []
best_model = None
best_auc = 0
best_name = ""

for name, config in models.items():
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"Description: {config['description']}")
    print('='*60)
    
    try:
        # Train model
        model = config['model']
        model.fit(X_train_resampled, y_train_resampled)
        
        # Predictions
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        
        # Metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_proba)
        
        # Cross-validation score
        cv_scores = cross_val_score(model, X_train_resampled, y_train_resampled, 
                                   cv=5, scoring='roc_auc')
        
        results.append({
            'Model': name,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1,
            'ROC-AUC': roc_auc,
            'CV Mean': cv_scores.mean(),
            'CV Std': cv_scores.std()
        })
        
        print(f"\n📊 Performance Metrics:")
        print(f"   Accuracy:  {accuracy:.4f}")
        print(f"   Precision: {precision:.4f}")
        print(f"   Recall:    {recall:.4f}")
        print(f"   F1-Score:  {f1:.4f}")
        print(f"   ROC-AUC:   {roc_auc:.4f}")
        print(f"\n📈 Cross-Validation:")
        print(f"   Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")
        
        if roc_auc > best_auc:
            best_auc = roc_auc
            best_model = model
            best_name = name
            
    except Exception as e:
        print(f"❌ Error training {name}: {e}")

# Step 5: Model comparison
print("\n📊 STEP 5: MODEL COMPARISON")
print("="*80)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('ROC-AUC', ascending=False)

print("\n🏆 Model Performance Ranking:")
print(results_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']].to_string(index=False))

print(f"\n🥇 Best Model: {best_name}")
print(f"   ROC-AUC Score: {best_auc:.4f}")

# Step 6: Save the best model
print("\n💾 STEP 6: SAVING BEST MODEL")
print("-"*50)

joblib.dump(best_model, 'models/churn_model.pkl')
print(f"✅ Best model saved to: models/churn_model.pkl")
print(f"   Model Type: {best_name}")

# Step 7: Feature importance (for tree-based models)
if hasattr(best_model, 'feature_importances_'):
    print("\n🔍 STEP 7: FEATURE IMPORTANCE ANALYSIS")
    print("-"*50)
    
    feature_importance = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\nTop 10 Most Important Features:")
    print(feature_importance.head(10).to_string(index=False))
    
    # Save feature importance
    feature_importance.to_csv('data/processed/model_feature_importance.csv', index=False)
    print("✅ Feature importance saved to: data/processed/model_feature_importance.csv")

# Step 8: Save model metadata
print("\n📝 STEP 8: SAVING MODEL METADATA")
print("-"*50)

model_metadata = {
    'best_model_name': best_name,
    'best_roc_auc': best_auc,
    'training_samples': len(X_train_resampled),
    'features_count': X_train.shape[1],
    'algorithm': best_name
}

joblib.dump(model_metadata, 'models/model_metadata.pkl')
print("✅ Model metadata saved to: models/model_metadata.pkl")

print("\n" + "="*80)
print("✅ TASKS 4 & 5 COMPLETED: MODEL SELECTION & TRAINING")
print("="*80)
print("\n📝 Summary of Actions:")
print(f"1. ✅ Evaluated {len(models)} binary classification algorithms")
print(f"2. ✅ Used SMOTE to handle class imbalance")
print(f"3. ✅ Selected {best_name} as the best model")
print(f"4. ✅ Trained model with ROC-AUC: {best_auc:.4f}")
print("5. ✅ Saved trained model and metadata")
print("\n🎯 Best Model Characteristics:")
if best_name == 'Random Forest':
    print("   - Handles non-linear relationships well")
    print("   - Provides feature importance")
    print("   - Robust to outliers")
elif best_name == 'Gradient Boosting':
    print("   - Sequential learning from errors")
    print("   - Often provides highest accuracy")
    print("   - Handles complex patterns")
elif best_name == 'Logistic Regression':
    print("   - Highly interpretable")
    print("   - Fast training and inference")
    print("   - Good baseline model")
elif best_name == 'SVM':
    print("   - Effective in high-dimensional spaces")
    print("   - Uses kernel tricks for non-linear data")
    print("   - Memory efficient")
print("\n🚀 Ready for next step: Model Evaluation (Task 6)")

# Add this at the end of 4_model_training.ipynb
import os
import joblib

# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# Save the model
joblib.dump(best_model, 'models/churn_model.pkl')
print("✅ Model saved to: models/churn_model.pkl")

# Also save the feature names for the app
feature_names = X_train.columns.tolist()
joblib.dump(feature_names, 'models/feature_names.pkl')
print("✅ Feature names saved to: models/feature_names.pkl")

# Save model metadata
model_metadata = {
    'best_model_name': best_name,
    'best_roc_auc': best_auc,
    'features_count': X_train.shape[1]
}
joblib.dump(model_metadata, 'models/model_metadata.pkl')
print("✅ Model metadata saved to: models/model_metadata.pkl")

# Print confirmation
print("\n" + "="*60)
print("FILES SAVED:")
print("="*60)
print("1. models/churn_model.pkl")
print("2. models/feature_names.pkl")
print("3. models/model_metadata.pkl")
print("="*60)

TASKS 4 & 5: MODEL SELECTION & TRAINING
Binary Classification Algorithms for Churn Prediction

📂 STEP 1: LOADING TRAINING DATA
--------------------------------------------------
✅ Data loaded successfully!
   Training set: (5625, 35)
   Testing set: (1407, 35)

⚖️ STEP 2: HANDLING CLASS IMBALANCE
--------------------------------------------------
Before SMOTE:
   Churn: 1495 (26.6%)
   No Churn: 4130 (73.4%)

After SMOTE:
   Churn: 4130 (50.0%)
   No Churn: 4130 (50.0%)

🤖 STEP 3: SELECTING BINARY CLASSIFICATION ALGORITHMS
--------------------------------------------------

🚀 STEP 4: TRAINING MODELS

Training: Logistic Regression
Description: Baseline model, interpretable, good for linear relationships

📊 Performance Metrics:
   Accuracy:  0.7605
   Precision: 0.5421
   Recall:    0.6364
   F1-Score:  0.5855
   ROC-AUC:   0.8126

📈 Cross-Validation:
   Mean: 0.9109 (+/- 0.1254)

Training: Random Forest
Description: Ensemble method, handles non-linear relationships, feature importance

